In [ ]:
VERSION = "1.0.4_sextractor"

# "1.0.1": "ellips gaussian + pixels_out",
# "1.0.2": "ellips uniform + pixels_out",
# "1.0.3": "ellips gaussian",
# hypercube + 40_000 + shear round 2 + 10 gal/patch + shear |0.06|

# lib

In [ ]:
from pathlib import Path

from jax import numpy
from polars import DataFrame, read_csv
from tqdm import tqdm
from utils import (
    best_results,
    case_results,
    ellips_ab,
    ellips_quad,
    ellips_ratio,
    extra_imgs,
    histos,
    vignettes,
)

DATASET_DIR = Path("/net/GECO/pisco/PIX2SHEAR/DATASETS/pisco_euclid") / VERSION
VERSION_DIR = Path("/net/GECO/pisco/PIX2SHEAR/RESULTS_SEXTRACTOR") / VERSION
CATALOG_DIR = VERSION_DIR / "CATALOG"
IMAGES_DIR = VERSION_DIR / "IMAGES"
PLOTS_DIR = VERSION_DIR / "PLOTS"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# images

In [ ]:
CHOICE = "simu_00002"

extra_imgs(
    [
        DATASET_DIR / CHOICE,
        IMAGES_DIR / f"fitting_{CHOICE}_1",
        IMAGES_DIR / f"background_{CHOICE}",
        IMAGES_DIR / f"residual_{CHOICE}_1",
        IMAGES_DIR / f"segmentation_{CHOICE}",
    ],
    ["image", "fitting", "background", "residual", "segmentation"],
    PLOTS_DIR,
)

# vignettes

In [ ]:
CHOICE = "simu_00001"

vignettes(
    PLOTS_DIR,
    csv_file=CATALOG_DIR / f"catalog_{CHOICE}.csv",
    vignettes_file=IMAGES_DIR / f"vignettes_{CHOICE}.fits",
)

# inference

In [ ]:
RESULTS_FILES = sorted(list((CATALOG_DIR).glob("catalog_simu_*.csv")))
NB_TOTAL = len(RESULTS_FILES)

detections, avg_e1_ra, avg_e2_ra, avg_e1, avg_e2, avg_e1_ab, avg_e2_ab = [], [], [], [], [], [], []
avg_e1_quad, avg_e2_quad = [], []
for sample in tqdm(RESULTS_FILES, total=NB_TOTAL):
    df = read_csv(sample)
    detections.append(df.shape[0])

    e1 = df["e1"].to_numpy()
    e2 = df["e2"].to_numpy()
    avg_e1.append(e1.mean())
    avg_e2.append(e2.mean())

    ratio = df["ratio"].to_numpy()
    angle = df["angle"].to_numpy()
    e1_ra, e2_ra = ellips_ratio(ratio, angle)
    avg_e1_ra.append(e1_ra.mean())
    avg_e2_ra.append(e2_ra.mean())

    a = df["ellipse_a"].to_numpy()
    b = df["ellipse_b"].to_numpy()
    theta = df["ellipse_theta"].to_numpy()
    e1_ab, e2_ab = ellips_ab(a, b, theta)
    avg_e1_ab.append(e1_ab.mean())
    avg_e2_ab.append(e2_ab.mean())

    ellipse_cxx = df["ellipse_cxx"].to_numpy()
    ellipse_cyy = df["ellipse_cyy"].to_numpy()
    ellipse_cxy = df["ellipse_cxy"].to_numpy()
    e1_quad, e2_quad = ellips_quad(ellipse_cxx, ellipse_cyy, ellipse_cxy)
    avg_e1_quad.append(e1_ab.mean())
    avg_e2_quad.append(e2_ab.mean())

# save res

In [ ]:
DataFrame(
    {
        "avg_e1": avg_e1,
        "avg_e2": avg_e2,
        "avg_e1_ra": avg_e1_ra,
        "avg_e2_ra": avg_e2_ra,
        "avg_e1_ab": avg_e1_ab,
        "avg_e2_ab": avg_e2_ab,
        "avg_e1_quad": avg_e1_quad,
        "avg_e2_quad": avg_e2_quad,
    },
    schema=[
        "avg_e1",
        "avg_e2",
        "avg_e1_ra",
        "avg_e2_ra",
        "avg_e1_ab",
        "avg_e2_ab",
        "avg_e1_quad",
        "avg_e2_quad",
    ],
).write_csv(VERSION_DIR / "pred_avg_ellips.csv")

del df, avg_e1, avg_e2, avg_e1_ra, avg_e2_ra, avg_e1_ab, avg_e2_ab, e1, e2, ratio, angle
del a, b, theta, e1_ra, e2_ra, e1_ab, e2_ab, ellipse_cxx, ellipse_cyy, ellipse_cxy, e1_quad, e2_quad

# postprocess

In [ ]:
pred_avg_e = read_csv(VERSION_DIR / "pred_avg_ellips.csv", separator=",", skip_rows=0).to_numpy()
true_avg_e = read_csv(DATASET_DIR / "true_avg_ellips.csv", separator=",", skip_rows=0).to_numpy()
true_shears = read_csv(DATASET_DIR / "true_avg_shears.csv", separator=",", skip_rows=0).to_numpy()
print(pred_avg_e.shape, true_avg_e.shape, true_shears.shape)

for i in range(2):
    pred_avg_e = pred_avg_e[numpy.abs(true_shears[:, i]) <= 0.05]
    true_avg_e = true_avg_e[numpy.abs(true_shears[:, i]) <= 0.05]
    true_shears = true_shears[numpy.abs(true_shears[:, i]) <= 0.05]
print("restricted range", pred_avg_e.shape, true_avg_e.shape, true_shears.shape)

# histo

In [ ]:
bins = 10
histos(
    diff_e1=pred_avg_e[:, 0] - pred_avg_e[:, 2],
    diff_e2=pred_avg_e[:, 1] - pred_avg_e[:, 3],
    data_gal=detections,
    directory=PLOTS_DIR / "errors_ra.pdf",
    bins=bins,
)
histos(
    diff_e1=pred_avg_e[:, 4] - pred_avg_e[:, 6],
    diff_e2=pred_avg_e[:, 5] - pred_avg_e[:, 7],
    data_gal=detections,
    directory=PLOTS_DIR / "errors_ab.pdf",
    bins=bins,
)

# results

In [ ]:
best_results(true_avg_e, true_shears, PLOTS_DIR / "best_results.pdf")

In [ ]:
case = "normal"
case_results(pred_avg_e[:, 0:2], true_shears, PLOTS_DIR / f"results_{case}.pdf")

case = "ratio_angle"
case_results(pred_avg_e[:, 2:4], true_shears, PLOTS_DIR / f"results_{case}.pdf")

In [ ]:
case = "ab_theta"
case_results(pred_avg_e[:, 4:6], true_shears, PLOTS_DIR / f"results_{case}.pdf")

case = "quad"
case_results(pred_avg_e[:, 6:8], true_shears, PLOTS_DIR / f"results_{case}.pdf")

# sersic (numerical stability?)

In [ ]:
# from utils import plot_sersic
# X_min, X_max = -2.5, 1.5
# plot_sersic(X_min, X_max, PLOTS_DIR)
# del X_min, X_max

# psf

In [ ]:
# from utils import check_psf
# check_psf(
#     Path("/net/GECO/pisco/PIX2SHEAR/DATASETS/pisco_euclid") / "PSF" / "airy.psf.fits", PLOTS_DIR
# )